In [2]:
from pymatgen.analysis.local_env import CrystalNN
from pymatgen.core import Composition, Structure
from pymatgen.io.cif import CifParser
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import pickle
import gzip

In [3]:
with gzip.open("val_cifs_1000.pkl.gz", "rb") as f:
    cif_pairs = pickle.load(f)

In [6]:
ex = cif_pairs[0][1]

In [8]:
parser = CifParser.from_string(ex)

In [9]:
cif_data = parser.as_dict()

In [16]:
import re

def extract_data_formula(cif_str):
    match = re.search(r"data_([A-Za-z0-9]+)\n", cif_str)
    if match:
        return match.group(1)
    raise Exception(f"could not find data_ in:\n{cif_str}")

In [17]:
formula_data = Composition(extract_data_formula(ex))

In [19]:
formula_sum = Composition(cif_data[list(cif_data.keys())[0]]["_chemical_formula_sum"])

In [22]:
formula_structural = Composition(cif_data[list(cif_data.keys())[0]]["_chemical_formula_structural"])

In [34]:
from collections import defaultdict

In [46]:
 # Extract the chemical formula sum from the CIF data
formula_sum = cif_data[list(cif_data.keys())[0]]["_chemical_formula_sum"]

# Convert the formula sum into a dictionary
expected_atoms = Composition(formula_sum).as_dict()

# Count the atoms provided in the _atom_site_type_symbol section
actual_atoms = defaultdict(int)
for key in cif_data:
    if "_atom_site_type_symbol" in cif_data[key] and "_atom_site_symmetry_multiplicity" in cif_data[key]:
        for atom_type, multiplicity in zip(cif_data[key]["_atom_site_type_symbol"],
                                            cif_data[key]["_atom_site_symmetry_multiplicity"]):
            # print(f"atom type: {atom_type}")
            # print(f"multiplicity: {multiplicity}")
            actual_atoms[atom_type] += int(multiplicity)
            # if atom_type in actual_atoms:
            #     actual_atoms[atom_type] += int(multiplicity)
            # else:
            #     actual_atoms[atom_type] = int(multiplicity)

In [51]:
structure = Structure.from_str(ex, fmt="cif")

In [56]:
# Extract the stated space group from the CIF file
stated_space_group = cif_data[list(cif_data.keys())[0]]['_symmetry_space_group_name_H-M']

# Analyze the symmetry of the structure
spacegroup_analyzer = SpacegroupAnalyzer(structure, symprec=0.1)

# Get the detected space group
detected_space_group = spacegroup_analyzer.get_space_group_symbol()

# Check if the detected space group matches the stated space group
is_match = stated_space_group.strip() == detected_space_group.strip()


c:\Users\amrut\anaconda3\envs\crystallm\Lib\site-packages\pymatgen\symmetry\analyzer.py:111: DeprecationWarning: dict interface is deprecated. Use attribute interface instead
  return self._space_group_data["international"]
